# 05. Breitensuche

In [Notebook 21 von learn_jwinf3](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_jwinf3/21_erreichbarkeit.ipynb) hast du mit einer Liste als Stapel
herausgefunden, **welche** Felder erreichbar sind. Jetzt geht es um die Frage: **Wie weit** ist es bis dorthin, und welcher
Weg ist der **kürzeste**? Die Antwort gibt die **Breitensuche** (englisch *breadth-first search*, BFS). Sie gehört zu den
wichtigsten Werkzeugen der 1. Runde: Labyrinthe, Netze, Rätsel mit Zügen, überall, wo jeder Schritt gleich viel kostet.

In diesem Notebook lernst du:

- einen **Graphen** als Wörterbuch von Nachbarlisten zu speichern,
- die Breitensuche mit einer **Warteschlange** (`deque`),
- warum man einen Knoten schon beim **Einreihen** als besucht markiert,
- **Vorgänger** zu merken und den kürzesten **Weg** zurückzuverfolgen,
- ein **Raster** als Graphen zu behandeln, ohne das Wörterbuch aufzuschreiben.

In der Wettbewerbs-Ecke hilfst du Ron in der Zauberschule und legst Linien in Arukone-Rätsel.

## 1. Ein Graph als Wörterbuch

Ein **Graph** besteht aus **Knoten** und **Kanten**, die je zwei Knoten verbinden. Im U-Bahn-Netz einer Stadt sind die
Stationen die Knoten, und zwei Stationen sind durch eine Kante verbunden, wenn eine Linie direkt von der einen zur anderen
fährt. Das Netz hat vier Linien:

| Linie | Stationen |
|---|---|
| U1 | Hafen – Markt – Dom – Uni – Zoo |
| U2 | Bahnhof – Markt – Rathaus – Park |
| U3 | Zoo – Park – Stadion – Messe |
| U4 | Dom – Rathaus – Stadion |

Wir speichern den Graphen als **Wörterbuch**: Zu jeder Station gehört die Liste ihrer Nachbarn. Weil man in beide Richtungen
fahren kann, steht jede Kante zweimal darin: Markt ist Nachbar von Hafen und Hafen ist Nachbar von Markt.

#### Aufgabe 1.1

Schreibe `baue_graph(linien)`, die aus der Liste der Linien das Wörterbuch macht. Jede Linie ist eine Liste von Stationen,
benachbarte Stationen sind verbunden.

**Erwartete Ausgabe:**
```
10 Stationen
Markt ['Hafen', 'Dom', 'Bahnhof', 'Rathaus']
Zoo ['Uni', 'Park']
Messe ['Stadion']
```

In [ ]:
linien = [['Hafen', 'Markt', 'Dom', 'Uni', 'Zoo'],
          ['Bahnhof', 'Markt', 'Rathaus', 'Park'],
          ['Zoo', 'Park', 'Stadion', 'Messe'],
          ['Dom', 'Rathaus', 'Stadion']]

def baue_graph(linien):
    graph = {}
    # dein Code
    return graph

netz = baue_graph(linien)
print(len(netz), 'Stationen')
for station in ['Markt', 'Zoo', 'Messe']:
    if station in netz:
        print(station, netz[station])

<details>
<summary>Tipp anzeigen</summary>

Erst jede Station mit einer leeren Liste eintragen, wenn sie noch nicht im Wörterbuch ist. Dann für jedes Paar
`linie[i]`, `linie[i + 1]` beide Richtungen anhängen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
linien = [['Hafen', 'Markt', 'Dom', 'Uni', 'Zoo'],
          ['Bahnhof', 'Markt', 'Rathaus', 'Park'],
          ['Zoo', 'Park', 'Stadion', 'Messe'],
          ['Dom', 'Rathaus', 'Stadion']]

def baue_graph(linien):
    graph = {}
    for linie in linien:
        for station in linie:
            if station not in graph:
                graph[station] = []
        for i in range(len(linie) - 1):
            a = linie[i]
            b = linie[i + 1]
            graph[a].append(b)               # beide Richtungen eintragen
            graph[b].append(a)
    return graph

netz = baue_graph(linien)
print(len(netz), 'Stationen')
for station in ['Markt', 'Zoo', 'Messe']:
    if station in netz:
        print(station, netz[station])
```

</details>

## 2. Die Breitensuche

Die Idee: Vom Start aus erst alle Knoten im Abstand 1 finden, dann alle im Abstand 2 und so weiter, wie Wellen im Wasser.
Dafür braucht man eine **Warteschlange**: Wer zuerst entdeckt wird, wird zuerst weiterbearbeitet.

1. `abstand = {start: 0}`, die Warteschlange enthält nur den Start.
2. Solange die Warteschlange nicht leer ist: den **vordersten** Knoten `v` herausnehmen. Für jeden Nachbarn `u`, der noch
   **nicht** in `abstand` steht: `abstand[u] = abstand[v] + 1` und `u` **hinten** anstellen.

Das Wörterbuch `abstand` dient zugleich als Liste der besuchten Knoten. Wichtig: Ein Knoten bekommt seinen Abstand, sobald
er **entdeckt** und eingereiht wird, nicht erst, wenn er herauskommt. So kommt jeder Knoten nur einmal in die Schlange.

**`deque`.** Mit einer normalen Liste wäre `liste.pop(0)` (vorne herausnehmen) langsam, denn alle anderen Elemente rücken
nach. Eine `deque` aus `collections` („double-ended queue“, sprich *deck*) kann vorne und hinten schnell herausnehmen und
anhängen: `schlange.append(x)` hinten anstellen, `schlange.popleft()` vorne herausnehmen.

#### Aufgabe 2.1

Schreibe `abstaende(graph, start)`, die das Wörterbuch der Abstände (Zahl der Fahrten) von `start` zu allen erreichbaren
Stationen zurückgibt.

**Erwartete Ausgabe:**
```
{'Hafen': 0, 'Markt': 1, 'Dom': 2, 'Bahnhof': 2, 'Rathaus': 2, 'Uni': 3, 'Park': 3, 'Stadion': 3, 'Zoo': 4, 'Messe': 4}
```

In [ ]:
from collections import deque

def abstaende(graph, start):
    abstand = {start: 0}
    schlange = deque([start])
    # dein Code
    return abstand

ab_hafen = abstaende(netz, 'Hafen')
print(ab_hafen)

<details>
<summary>Tipp anzeigen</summary>

`while len(schlange) > 0:`, darin `v = schlange.popleft()` und für jeden Nachbarn `u` in `graph[v]`, der nicht in `abstand`
ist: Abstand eintragen und `schlange.append(u)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from collections import deque

def abstaende(graph, start):
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0:
        v = schlange.popleft()                   # vorne herausnehmen
        for u in graph[v]:
            if u not in abstand:                 # noch nicht entdeckt
                abstand[u] = abstand[v] + 1      # gleich beim Einreihen merken
                schlange.append(u)               # hinten anstellen
    return abstand

ab_hafen = abstaende(netz, 'Hafen')
print(ab_hafen)
```

Die Stationen stehen im Wörterbuch in der Reihenfolge, in der sie entdeckt wurden: erst Abstand 0, dann alle mit 1, dann
alle mit 2 und so weiter. Genau das ist die Breitensuche.

</details>

#### Frage 2.2

1. Warum ist der Abstand, den ein Knoten beim Entdecken bekommt, sicher der **kürzeste**?
2. Wie lange braucht die Breitensuche für einen Graphen mit `V` Knoten und `E` Kanten?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Warteschlange enthält immer erst alle Knoten mit Abstand `d` und dahinter die mit `d + 1`. Darum werden die Knoten
   nach ihrem Abstand geordnet herausgenommen. Ein Knoten wird zum ersten Mal von einem Nachbarn mit dem kleinsten
   möglichen Abstand entdeckt. Gäbe es einen kürzeren Weg, wäre sein vorletzter Knoten früher herausgekommen und hätte den
   Knoten schon vorher entdeckt.
2. Jeder Knoten kommt einmal in die Schlange und einmal heraus, und bei jedem werden seine Nachbarn einmal angesehen. Jede
   Kante wird also zweimal betrachtet (von jedem Ende einmal). Zusammen ist das **O(V + E)**, so schnell, wie man den Graphen
   überhaupt lesen kann. Ein Graph mit einer Million Knoten ist damit in Sekunden durchsucht.

</details>

## 3. Den Weg zurückverfolgen

Der Abstand allein sagt nicht, **wie** man fährt. Dafür merkt man sich zu jedem Knoten, von welchem Knoten aus er entdeckt
wurde: seinen **Vorgänger**. Am Ziel geht man dann von Vorgänger zu Vorgänger zurück bis zum Start und dreht die Liste um.
Das Wörterbuch `vorgaenger` kann dabei `abstand` als Liste der besuchten Knoten ersetzen.

Man kann aufhören, sobald das Ziel entdeckt ist. Ist die Schlange leer, ohne dass das Ziel entdeckt wurde, ist es nicht
erreichbar.

#### Aufgabe 3.1

Schreibe `kuerzester_weg(graph, start, ziel)`, die die Liste der Stationen eines kürzesten Weges zurückgibt, oder `None`, wenn
das Ziel nicht erreichbar ist.

**Erwartete Ausgabe:**
```
['Hafen', 'Markt', 'Rathaus', 'Stadion', 'Messe']
['Bahnhof', 'Markt', 'Dom', 'Uni', 'Zoo']
None
```

In [ ]:
def kuerzester_weg(graph, start, ziel):
    vorgaenger = {start: None}
    schlange = deque([start])
    # dein Code
    return None

print(kuerzester_weg(netz, 'Hafen', 'Messe'))
print(kuerzester_weg(netz, 'Bahnhof', 'Zoo'))
netz['Insel'] = []                               # eine Station ohne Verbindung
print(kuerzester_weg(netz, 'Hafen', 'Insel'))

<details>
<summary>Tipp anzeigen</summary>

Die Schleife wie in 2.1, nur mit `vorgaenger[u] = v` statt des Abstands, und mit der zusätzlichen Bedingung
`ziel not in vorgaenger`. Danach vom Ziel aus `knoten = vorgaenger[knoten]`, bis `None` kommt, und die Liste mit `reverse()`
umdrehen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def kuerzester_weg(graph, start, ziel):
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in vorgaenger:
        v = schlange.popleft()
        for u in graph[v]:
            if u not in vorgaenger:
                vorgaenger[u] = v                # u wurde von v aus entdeckt
                schlange.append(u)
    if ziel not in vorgaenger:
        return None                              # nicht erreichbar
    weg = []
    knoten = ziel
    while knoten is not None:                    # rückwärts bis zum Start
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return weg

print(kuerzester_weg(netz, 'Hafen', 'Messe'))
print(kuerzester_weg(netz, 'Bahnhof', 'Zoo'))
netz['Insel'] = []                               # eine Station ohne Verbindung
print(kuerzester_weg(netz, 'Hafen', 'Insel'))
```

</details>

#### Frage 3.2

Die Breitensuche findet einen Weg mit den wenigsten **Fahrten**. Fahrgäste wollen aber oft möglichst wenig **umsteigen**.
Ist der Weg von Hafen nach Messe auch der mit den wenigsten Umstiegen? Wie könnte man nach Umstiegen suchen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Der gefundene Weg Hafen – Markt – Rathaus – Stadion – Messe hat 4 Fahrten, aber er benutzt die Linien U1, U2, U4 und U3, also
**drei** Umstiege. Hafen – Markt – Dom – Uni – Zoo – Park – Stadion – Messe hat 7 Fahrten, aber nur einen Umstieg (U1, dann
U3).

Um nach Umstiegen zu suchen, ändert man, was ein Knoten ist: nicht nur die Station, sondern das Paar **(Station, Linie)**.
Weiterfahren auf derselben Linie kostet keinen Umstieg, Umsteigen kostet einen. Solche Knoten aus mehreren Angaben heißen
**Zustände**, sie sind das Thema von Notebook 06. Weil Fahrten und Umstiege verschieden viel kosten, braucht man dann
Notebook 07 (Dijkstra).

</details>

## 4. Ein Raster als Graph

In einem Labyrinth ist jedes freie Feld `(zeile, spalte)` ein Knoten, und benachbart sind die bis zu vier Felder oben, unten,
links und rechts, die keine Wand `#` sind. Das Wörterbuch muss man dafür nicht aufschreiben: Eine Funktion
`nachbarfelder(raster, feld)` berechnet die Nachbarn erst, wenn sie gebraucht werden. Die Breitensuche bleibt genau gleich.

Die Datei `daten/labyrinth.txt` enthält ein Labyrinth mit Start `S` und Ziel `Z`. Das Raster speichern wir als Liste von
Texten, `raster[zeile][spalte]` ist ein Zeichen.

#### Aufgabe 4.1

Schreibe `nachbarfelder(raster, feld)`. Sie gibt die Liste der Nachbarfelder zurück, die im Raster liegen und keine Wand sind.

**Erwartete Ausgabe:**
```
#####################
#S....#...........#.#
#####.#.#.#######.#.#
#...#.#.#.#.......#.#
###.#.###.#####.###.#
#...#...#.....#...#.#
#.#####.#####.###.#.#
#...........#.......#
#.#.#.#.###.###.#.#.#
#...#...........#..Z#
#####################
(1, 1) (9, 19)
[(1, 2)] [(3, 2), (4, 3)]
```

In [ ]:
RICHTUNGEN = [(0, 1), (0, -1), (1, 0), (-1, 0)]          # rechts, links, unten, oben

def lies_raster(dateiname):
    raster = []
    for zeile in open(dateiname):
        if zeile.strip() != '':
            raster.append(zeile.rstrip('\n'))
    return raster

def finde(raster, zeichen):
    for r in range(len(raster)):
        for c in range(len(raster[r])):
            if raster[r][c] == zeichen:
                return (r, c)
    return None

def nachbarfelder(raster, feld):
    liste = []
    # dein Code
    return liste

labyrinth = lies_raster('daten/labyrinth.txt')
for zeile in labyrinth:
    print(zeile)
print(finde(labyrinth, 'S'), finde(labyrinth, 'Z'))
print(nachbarfelder(labyrinth, (1, 1)), nachbarfelder(labyrinth, (3, 3)))

<details>
<summary>Tipp anzeigen</summary>

Für jede Richtung `(dz, ds)` das Feld `(feld[0] + dz, feld[1] + ds)` berechnen und anhängen, wenn es im Raster liegt
(`0 <= r < len(raster)` und `0 <= c < len(raster[r])`) und `raster[r][c] != '#'` ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
RICHTUNGEN = [(0, 1), (0, -1), (1, 0), (-1, 0)]          # rechts, links, unten, oben

def lies_raster(dateiname):
    raster = []
    for zeile in open(dateiname):
        if zeile.strip() != '':
            raster.append(zeile.rstrip('\n'))
    return raster

def finde(raster, zeichen):
    for r in range(len(raster)):
        for c in range(len(raster[r])):
            if raster[r][c] == zeichen:
                return (r, c)
    return None

def nachbarfelder(raster, feld):
    liste = []
    for dz, ds in RICHTUNGEN:
        r = feld[0] + dz
        c = feld[1] + ds
        if 0 <= r < len(raster) and 0 <= c < len(raster[r]) and raster[r][c] != '#':
            liste.append((r, c))
    return liste

labyrinth = lies_raster('daten/labyrinth.txt')
for zeile in labyrinth:
    print(zeile)
print(finde(labyrinth, 'S'), finde(labyrinth, 'Z'))
print(nachbarfelder(labyrinth, (1, 1)), nachbarfelder(labyrinth, (3, 3)))
```

</details>

#### Aufgabe 4.2

Schreibe `weg_im_raster(raster, start, ziel)` wie `kuerzester_weg`, aber mit `nachbarfelder`. Sie soll den Weg und die Zahl
der entdeckten Felder (`len(vorgaenger)`) zurückgeben. Zeichne den Weg mit der fertigen Funktion `zeichne` ein.

**Erwartete Ausgabe:**
```
30 Schritte, 70 Felder entdeckt
#####################
#S****#...........#.#
#####*#.#.#######.#.#
#...#*#.#.#.......#.#
###.#*###.#####.###.#
#...#***#.....#...#.#
#.#####*#####.###.#.#
#......*****#..*****#
#.#.#.#.###*###*#.#*#
#...#......*****#..Z#
#####################
```

In [ ]:
def weg_im_raster(raster, start, ziel):
    vorgaenger = {start: None}
    schlange = deque([start])
    # dein Code
    return None, len(vorgaenger)

def zeichne(raster, weg, zeichen):
    zeilen = []
    for zeile in raster:
        zeilen.append(list(zeile))
    for k in range(1, len(weg) - 1):                 # Start und Ziel bleiben stehen
        r, c = weg[k]
        zeilen[r][c] = zeichen[k]
    for zeile in zeilen:
        print(''.join(zeile))

weg_lab, entdeckt = weg_im_raster(labyrinth, finde(labyrinth, 'S'), finde(labyrinth, 'Z'))
if weg_lab is not None:
    print(len(weg_lab) - 1, 'Schritte,', entdeckt, 'Felder entdeckt')
    zeichne(labyrinth, weg_lab, '*' * len(weg_lab))

<details>
<summary>Tipp anzeigen</summary>

Genau wie `kuerzester_weg`, nur `for u in nachbarfelder(raster, v):` statt `for u in graph[v]:`, und zurückgegeben werden zwei
Werte.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def weg_im_raster(raster, start, ziel):
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in vorgaenger:
        v = schlange.popleft()
        for u in nachbarfelder(raster, v):
            if u not in vorgaenger:
                vorgaenger[u] = v
                schlange.append(u)
    if ziel not in vorgaenger:
        return None, len(vorgaenger)
    weg = []
    knoten = ziel
    while knoten is not None:
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return weg, len(vorgaenger)

def zeichne(raster, weg, zeichen):
    zeilen = []
    for zeile in raster:
        zeilen.append(list(zeile))
    for k in range(1, len(weg) - 1):                 # Start und Ziel bleiben stehen
        r, c = weg[k]
        zeilen[r][c] = zeichen[k]
    for zeile in zeilen:
        print(''.join(zeile))

weg_lab, entdeckt = weg_im_raster(labyrinth, finde(labyrinth, 'S'), finde(labyrinth, 'Z'))
if weg_lab is not None:
    print(len(weg_lab) - 1, 'Schritte,', entdeckt, 'Felder entdeckt')
    zeichne(labyrinth, weg_lab, '*' * len(weg_lab))
```

Der Weg hat `len(weg) - 1` Schritte, denn die Liste enthält Start und Ziel. `zeichne` bekommt für jede Stelle des Weges ein
Zeichen. Hier sind es lauter Sterne, in der Wettbewerbs-Ecke werden es Pfeile.

</details>

## 5. Zwei typische Fehler

Für beide Fehler nehmen wir einen leeren Raum aus 20 × 20 Feldern ohne Wände. Der kürzeste Weg von der Ecke oben links zur
Ecke unten rechts hat 19 + 19 = 38 Schritte.

In [ ]:
raum = ['#' * 22]
for k in range(20):
    raum.append('#' + '.' * 20 + '#')
raum.append('#' * 22)

#### Frage 5.1

Finde den Fehler. Ben hat sich vertippt:

```python
def abstand_falsch(raster, start, ziel):
    abstand = {start: 0}
    schlange = deque([start])
    while len(schlange) > 0:
        v = schlange.pop()
        for u in nachbarfelder(raster, v):
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                schlange.append(u)
    return abstand[ziel]
```

`abstand_falsch(raum, (1, 1), (20, 20))` ergibt 190 statt 38. Warum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`schlange.pop()` nimmt das **letzte** Element heraus, nicht das erste. Dann ist die Schlange ein **Stapel**, und aus der
Breitensuche wird eine **Tiefensuche**: Sie läuft immer weiter in eine Richtung, bevor sie zurückkommt, wie in learn_jwinf3,
Notebook 21. Damit findet sie zwar alle erreichbaren Felder, aber die Abstände sind die Längen irgendwelcher Umwege, nicht
die kürzesten. Für kürzeste Wege braucht man `popleft()`.

</details>

#### Frage 5.2

Mia markiert ein Feld erst als besucht, wenn es aus der Schlange **herauskommt**:

```python
def zaehle_spaet(raster, start):
    besucht = set()
    schlange = deque([start])
    eingereiht = 1
    while len(schlange) > 0:
        v = schlange.popleft()
        if v in besucht:
            continue
        besucht.add(v)                   # erst beim Herausnehmen als besucht markiert
        for u in nachbarfelder(raster, v):
            if u not in besucht:
                schlange.append(u)
                eingereiht = eingereiht + 1
    return eingereiht
```

Für den Raum aus 400 Feldern zählt sie 761 Einreihungen. Was ist das Problem, und warum stimmt das Ergebnis
trotzdem?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Ein Feld kann mehrmals eingereiht werden, bevor es zum ersten Mal herauskommt: von jedem Nachbarn einmal, der es vorher
entdeckt. Im Raum haben die meisten Felder zwei Nachbarn mit kleinerem Abstand (links und oben), darum sind es fast doppelt
so viele Einreihungen wie Felder. Das `continue` sorgt dafür, dass jedes Feld trotzdem nur einmal bearbeitet wird. Das
Ergebnis stimmt also, aber das Programm macht unnötige Arbeit, und die Schlange wird länger. Bei Graphen mit vielen Kanten je
Knoten ist das viel schlimmer. Richtig ist: **beim Einreihen** markieren, wie in 2.1.

</details>

## Wettbewerbs-Ecke 1: Zauberschule auf einem Stockwerk

*Angelehnt an Aufgabe 3 „Zauberschule“ der 1. Runde des 42. Bundeswettbewerbs Informatik (2023/24).*

Die Zauberschule Bugwarts hat zwei Stockwerke mit Feldern und Wänden. Ron will so schnell wie möglich von Feld `A` nach Feld
`B`. Ein Schritt zum Nachbarfeld dauert 1 Sekunde, ein Wechsel ins andere Stockwerk 3 Sekunden.

Weil der Wechsel länger dauert als ein Schritt, reicht dafür die einfache Breitensuche nicht. Das lösen wir in Notebook 07.
Hier vereinfachen wir: **Ron bleibt im Stockwerk von `A`.** In allen Beispielen liegen `A` und `B` im oberen Stockwerk.

Die Dateien enthalten in der ersten Zeile die Größe `n m` eines Stockwerks, dann `n` Zeilen für das obere Stockwerk, eine
Leerzeile und `n` Zeilen für das untere. `lies_plan` ist fertig. Ron will den Weg mit Pfeilen sehen: `>` rechts, `<` links,
`v` hinunter, `^` hinauf.

#### Aufgabe E.1

Schreibe `pfeile(weg)`, die für jede Stelle des Weges das Zeichen der Richtung zurückgibt, in die Ron von dort weitergeht.
Löse damit `zauberschule0.txt` und zeichne den Weg ein.

**Erwartete Ausgabe:**
```
26 Sekunden, 57 Felder entdeckt
<<<<vv>>>>>>>>>>^^^^<<<<vvB
#############
#.....#.....#
#.###.#.###.#
#...#.#...#.#
###.#.###.#.#
#...#.....#.#
#.#########.#
#.....#v<<<<#
#####.#v###^#
#v<<<A#B..#^#
#v#########^#
#>>>>>>>>>>^#
#############
```

In [ ]:
def lies_plan(dateiname):
    zeilen = open(dateiname).read().split('\n')
    n = int(zeilen[0].split()[0])
    oben = zeilen[1:1 + n]
    unten = zeilen[2 + n:2 + 2 * n]
    return oben, unten

def pfeile(weg):
    zeichen = ''
    # dein Code
    return zeichen

oben0, unten0 = lies_plan('daten/zauberschule0.txt')
weg0, entdeckt0 = weg_im_raster(oben0, finde(oben0, 'A'), finde(oben0, 'B'))
print(len(weg0) - 1, 'Sekunden,', entdeckt0, 'Felder entdeckt')
print(pfeile(weg0))
if len(pfeile(weg0)) == len(weg0):
    zeichne(oben0, weg0, pfeile(weg0))

<details>
<summary>Tipp anzeigen</summary>

Für jedes `k` von 0 bis `len(weg) - 2` die Unterschiede `dz` und `ds` zwischen `weg[k + 1]` und `weg[k]` ausrechnen und das
passende Zeichen anhängen. Damit die Länge stimmt, hängst du am Ende noch ein Zeichen für das Ziel an.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_plan(dateiname):
    zeilen = open(dateiname).read().split('\n')
    n = int(zeilen[0].split()[0])
    oben = zeilen[1:1 + n]
    unten = zeilen[2 + n:2 + 2 * n]
    return oben, unten

def pfeile(weg):
    zeichen = ''
    for k in range(len(weg) - 1):
        dz = weg[k + 1][0] - weg[k][0]
        ds = weg[k + 1][1] - weg[k][1]
        if ds == 1:
            zeichen = zeichen + '>'
        elif ds == -1:
            zeichen = zeichen + '<'
        elif dz == 1:
            zeichen = zeichen + 'v'
        else:
            zeichen = zeichen + '^'
    return zeichen + 'B'                     # am Ziel geht es nicht weiter

oben0, unten0 = lies_plan('daten/zauberschule0.txt')
weg0, entdeckt0 = weg_im_raster(oben0, finde(oben0, 'A'), finde(oben0, 'B'))
print(len(weg0) - 1, 'Sekunden,', entdeckt0, 'Felder entdeckt')
print(pfeile(weg0))
if len(pfeile(weg0)) == len(weg0):
    zeichne(oben0, weg0, pfeile(weg0))
```

Ron muss im oberen Stockwerk einmal fast ganz herumlaufen, obwohl `B` direkt neben `A` liegt, nur durch eine Wand getrennt.

</details>

#### Aufgabe E.2

Bestimme für alle sechs Beispiele die Dauer im oberen Stockwerk und die Zahl der entdeckten Felder.

**Erwartete Ausgabe:**
```
zauberschule0 : 13 x 13 Felder, 26 Sekunden, 57 Felder entdeckt
zauberschule1 : 11 x 21 Felder, 4 Sekunden, 5 Felder entdeckt
zauberschule2 : 15 x 45 Felder, 32 Sekunden, 51 Felder entdeckt
zauberschule3 : 31 x 31 Felder, 28 Sekunden, 82 Felder entdeckt
zauberschule4 : 101 x 101 Felder, 674 Sekunden, 3003 Felder entdeckt
zauberschule5 : 201 x 81 Felder, 1030 Sekunden, 3815 Felder entdeckt
```

In [ ]:
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife über `nummer` in `range(6)`, darin `lies_plan`, `weg_im_raster` und eine Ausgabe.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for nummer in range(6):
    oben, unten = lies_plan('daten/zauberschule' + str(nummer) + '.txt')
    weg, entdeckt = weg_im_raster(oben, finde(oben, 'A'), finde(oben, 'B'))
    print('zauberschule' + str(nummer), ':', len(oben), 'x', len(oben[0]), 'Felder,', len(weg) - 1, 'Sekunden,', entdeckt,
          'Felder entdeckt')
```

Mit dem unteren Stockwerk geht es viel schneller. Laut Lösungsnotebook braucht Ron bei Beispiel 0 nur 8 statt 26 Sekunden,
bei Beispiel 4 nur 84 statt 674 und bei Beispiel 5 nur 124 statt 1030 Sekunden. Nur bei den Beispielen 1 und 3 lohnt sich
der Wechsel nicht. Wie man den Wechsel mit 3 Sekunden einbaut, lernst du in Notebook 07.

</details>

## Wettbewerbs-Ecke 2: Arukone (für Schnelle)

*Angelehnt an Aufgabe 1 „Arukone“ der 1. Runde des 42. Bundeswettbewerbs Informatik (2023/24).*

In einem Arukone-Rätsel steht jede Zahl genau zweimal im Gitter. Die beiden Felder jeder Zahl sollen durch einen Linienzug
aus waagerechten und senkrechten Schritten verbunden werden, und kein Feld darf zu zwei Linienzügen gehören. In der echten
Aufgabe soll man Rätsel **erfinden**, die ein „sehr einfaches“ Löseprogramm der BWINF nicht schafft. Die Lösungshinweise
vermuten, dass dieses Programm **gierig** vorgeht:

> Für die Zahlen 1, 2, 3, … nacheinander: den kürzesten Weg über freie Felder legen und nie wieder ändern.

Wir bauen diesen gierigen Löser und probieren ihn am Rätsel vom Aufgabenblatt aus (`daten/arukone0.txt`: erste Zeile Größe,
zweite Zeile Zahl der Paare, dann das Gitter mit 0 für leere Felder).

#### Aufgabe E.3

Schreibe `gierig(raetsel, reihenfolge)`. Sie kopiert das Gitter und verbindet die Zahlen in der gegebenen Reihenfolge mit
einem kürzesten Weg über Felder mit 0. Der Weg wird mit der Zahl gefüllt. Gibt es für eine Zahl keinen Weg, gibt die Funktion
diese Zahl zurück, sonst 0, und dazu das Gitter. Die Hilfsfunktion `weg_frei` ist die Breitensuche aus 4.2, aber nur über
freie Felder (und das Ziel).

**Erwartete Ausgabe:**
```
1 . 2 4 . . 
. . 3 . 5 . 
. . 1 . . . 
. . . . . 4 
. 2 . . . . 
. . . 3 . 5 

gescheitert bei 2
1 1 2 4 . . 
. 1 3 . 5 . 
. 1 1 . . . 
. . . . . 4 
. 2 . . . . 
. . . 3 . 5
```

In [ ]:
def lies_arukone(dateiname):
    zahlen = open(dateiname).read().split()
    n = int(zahlen[0])
    gitter = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append(int(zahlen[2 + r * n + c]))
        gitter.append(zeile)
    return gitter

def zeige_gitter(gitter):
    for zeile in gitter:
        text = ''
        for x in zeile:
            if x == 0:
                text = text + '. '
            else:
                text = text + str(x) + ' '
        print(text)

def weg_frei(gitter, start, ziel):
    n = len(gitter)
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in vorgaenger:
        v = schlange.popleft()
        for dz, ds in RICHTUNGEN:
            u = (v[0] + dz, v[1] + ds)
            if 0 <= u[0] < n and 0 <= u[1] < n and u not in vorgaenger and (gitter[u[0]][u[1]] == 0 or u == ziel):
                vorgaenger[u] = v
                schlange.append(u)
    if ziel not in vorgaenger:
        return None
    weg = []
    knoten = ziel
    while knoten is not None:
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return weg

def kopiere(gitter):
    neu = []
    for zeile in gitter:
        neu.append(list(zeile))
    return neu

def gierig(raetsel, reihenfolge):
    gitter = kopiere(raetsel)
    # dein Code
    return 0, gitter

arukone = lies_arukone('daten/arukone0.txt')
zeige_gitter(arukone)
print()
gescheitert, versuch = gierig(arukone, [1, 2, 3, 4, 5])
print('gescheitert bei', gescheitert)
zeige_gitter(versuch)

<details>
<summary>Tipp anzeigen</summary>

Für jede Zahl die beiden Felder suchen, in denen sie im Rätsel steht, dann `weg = weg_frei(gitter, felder[0], felder[1])`. Ist
`weg` `None`, gib `zahl, gitter` zurück. Sonst alle Felder des Weges auf `zahl` setzen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_arukone(dateiname):
    zahlen = open(dateiname).read().split()
    n = int(zahlen[0])
    gitter = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append(int(zahlen[2 + r * n + c]))
        gitter.append(zeile)
    return gitter

def zeige_gitter(gitter):
    for zeile in gitter:
        text = ''
        for x in zeile:
            if x == 0:
                text = text + '. '
            else:
                text = text + str(x) + ' '
        print(text)

def weg_frei(gitter, start, ziel):
    n = len(gitter)
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0 and ziel not in vorgaenger:
        v = schlange.popleft()
        for dz, ds in RICHTUNGEN:
            u = (v[0] + dz, v[1] + ds)
            if 0 <= u[0] < n and 0 <= u[1] < n and u not in vorgaenger and (gitter[u[0]][u[1]] == 0 or u == ziel):
                vorgaenger[u] = v
                schlange.append(u)
    if ziel not in vorgaenger:
        return None
    weg = []
    knoten = ziel
    while knoten is not None:
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return weg

def kopiere(gitter):
    neu = []
    for zeile in gitter:
        neu.append(list(zeile))
    return neu

def gierig(raetsel, reihenfolge):
    gitter = kopiere(raetsel)
    n = len(gitter)
    for zahl in reihenfolge:
        felder = []
        for r in range(n):
            for c in range(n):
                if raetsel[r][c] == zahl:
                    felder.append((r, c))
        weg = weg_frei(gitter, felder[0], felder[1])
        if weg is None:
            return zahl, gitter                  # kein freier Weg mehr
        for r, c in weg:
            gitter[r][c] = zahl
    return 0, gitter

arukone = lies_arukone('daten/arukone0.txt')
zeige_gitter(arukone)
print()
gescheitert, versuch = gierig(arukone, [1, 2, 3, 4, 5])
print('gescheitert bei', gescheitert)
zeige_gitter(versuch)
```

Der kürzeste Weg für die 1 läuft über das Feld links neben der oberen 2. Danach ist die obere 2 von der 1, der 3 und der 4
eingeschlossen. Ein gieriges Verfahren nimmt eine
Entscheidung nie zurück, auch wenn sie sich später als schlecht herausstellt.

</details>

#### Aufgabe E.4

Probiere mit `permutations` aus Notebook 01 alle Reihenfolgen der Zahlen 1 bis 5 aus. Bei wie vielen schafft der gierige
Löser das Rätsel? Gib eine Lösung aus.

**Erwartete Ausgabe:**
```
zum Beispiel mit der Reihenfolge (2, 1, 3, 4, 5)
1 2 2 4 4 4 
1 2 3 3 5 4 
1 2 1 3 5 4 
1 2 1 3 5 4 
1 2 1 3 5 5 
1 1 1 3 . 5 
geschafft mit 60 von 120 Reihenfolgen
```

In [ ]:
from itertools import permutations

# dein Code

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from itertools import permutations

geschafft = 0
for reihenfolge in permutations([1, 2, 3, 4, 5]):
    gescheitert, versuch = gierig(arukone, reihenfolge)
    if gescheitert == 0:
        if geschafft == 0:
            print('zum Beispiel mit der Reihenfolge', reihenfolge)
            zeige_gitter(versuch)
        geschafft = geschafft + 1
print('geschafft mit', geschafft, 'von 120 Reihenfolgen')
```

Das Rätsel ist also lösbar, aber der gierige Löser findet die Lösung nur mit der Hälfte der Reihenfolgen. Rätsel, die er mit
**keiner** Reihenfolge schafft, sind die, die die Aufgabe sucht. Das Lösungsnotebook erzeugt solche Rätsel und prüft sie mit
einem genauen Löser (Backtracking wie in Notebook 02).

</details>

## Das hast du gelernt

```python
from collections import deque

vorgaenger = {start: None}
schlange = deque([start])
while len(schlange) > 0 and ziel not in vorgaenger:
    v = schlange.popleft()                  # vorne heraus (nicht pop!)
    for u in nachbarn(v):
        if u not in vorgaenger:             # beim Einreihen markieren
            vorgaenger[u] = v
            schlange.append(u)
```

- **Graph** als Wörterbuch von Nachbarlisten, bei einem Raster eine Funktion `nachbarfelder`.
- **Breitensuche** findet kürzeste Wege, wenn **jeder Schritt gleich viel** kostet. Laufzeit O(V + E).
- **Vorgänger** merken, rückwärts gehen, Liste umdrehen.
- **Fehler:** `pop()` statt `popleft()` gibt eine Tiefensuche mit falschen Abständen. Spät markieren macht unnötige Arbeit.
- Kosten die Schritte verschieden viel (3 Sekunden für den Stockwerkswechsel), braucht man **Dijkstra** (Notebook 07).

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Zauberschule](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/42_Zauberschule/zauberschule_loesung.ipynb) (42. BwInf): beide Stockwerke
- [Arukone](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/42_Arukone/arukone_loesung.ipynb) (42.): Rätsel erzeugen, die der gierige Löser nicht schafft
- [Kassiopeias Weg](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/34_KassiopeiasWeg/kassiopeiasweg_loesung.ipynb) (34.): Zusammenhang prüfen
- [Telepaartie](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Telepaartie/telepaartie_loesung.ipynb) (38.) und
  [Faires Füllen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/33_FairesFuellen/fairesfuellen_loesung.ipynb) (33.): Breitensuche über Zustände (Notebook 06)

Im nächsten Notebook sind die Knoten keine Orte mehr, sondern **Zustände**: Wer steht wo, wie voll ist welches Gefäß?